### np.sort

In [3]:
def get_shape(a):
    if not isinstance(a , list): 
        return ()
    if len(a) == 0: 
        return (0 , )
    inner_shape = get_shape(a[0])

    for i in range(1 , len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected : the shape of first row is {inner_shape}" , 
                f"but row {i} has shape {get_shape(a[i])}"
            )

    return (len(a) , ) + inner_shape

def get_ndim(a):
    if not isinstance(a , list):
        return 0
    if len(a) == 0: 
        return 1
    return 1 + get_ndim(a[0])

def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result

def _product(shape):
    prod = 1
    for s in shape:
        prod *= s
    return prod 

def reshape(flat, shape):
    if len(shape) == 0:
        if len(flat) != 1:
            raise ValueError("Cannot reshape: size mismatch for scalar")
        return flat[0]

    size0 = shape[0]
    rest_shape = shape[1:]
    if len(rest_shape) == 0:
        if len(flat) != size0:
            raise ValueError("Cannot reshape: size mismatch")
        return list(flat)

    chunk_size = _product(rest_shape)
    if len(flat) != size0 * chunk_size:
        raise ValueError("Cannot reshape: size mismatch")

    result = []
    for i in range(size0):
        start = i * chunk_size
        end = (i + 1) * chunk_size
        chunk = flat[start:end]
        result.append(reshape(chunk, rest_shape))
    return result

def mergesort_1d(lst):
    n = len(lst)
    if n <= 1:
        return list(lst)

    mid = n // 2
    left = mergesort_1d(lst[:mid])
    right = mergesort_1d(lst[mid:])

    i = j = 0
    result = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    while i < len(left):
        result.append(left[i])
        i += 1
    while j < len(right):
        result.append(right[j])
        j += 1
    return result

def mergesort_1d(lst):
    if len(lst) <= 1:
        return lst[:]

    mid = len(lst) // 2
    left = mergesort_1d(lst[:mid])
    right = mergesort_1d(lst[mid:])

    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result


def sort_1d(lst, kind='mergesort'):
    if kind in ('mergesort', 'stable', 'quicksort', 'heapsort', None):
        return mergesort_1d(lst)
    raise ValueError(f"Unsupported sort kind: {kind}")

def np_sort(a, axis=-1, kind='mergesort'):
    
    shape = get_shape(a)
    ndim = len(shape)

   
    if ndim == 0:
        return a

    if axis is None:
        flat = flatten(a)
        flat_sorted = sort_1d(flat, kind)
        return reshape(flat_sorted, shape)

    if axis < 0:
        axis += ndim
    if axis < 0 or axis >= ndim:
        raise ValueError("axis out of range")

    if axis != ndim - 1:
        raise NotImplementedError("Only sorting along the last axis is implemented")

    return sort_last_axis(a, kind)

In [4]:
print(np_sort([3, 1, 2]))
print(np_sort(['b', 'a', 'c']))
print(np_sort([[3, 1], [2, 0]]))
print(np_sort([[[3, 1], [2, 0]], [[5, 4], [7, 6]]]))
print(np_sort([[3, 1], [2, 0]], axis=None))
print(np_sort([]))
print(np_sort(5))
print(np_sort([3, 1, 2], kind='mergesort'))
print(np_sort([3, 1, 2], kind='quicksort'))
print(np_sort([3, 1, 2], kind='stable'))


[1, 2, 3]
['a', 'b', 'c']
[[1, 3], [0, 2]]
[[[1, 3], [0, 2]], [[4, 5], [6, 7]]]
[[0, 1], [2, 3]]
[]
5
[1, 2, 3]
[1, 2, 3]
[1, 2, 3]


In [5]:
np_sort([[1, 2], [3]], axis=-1)

ValueError: ('Jagged array detected : the shape of first row is (2,)', 'but row 1 has shape (1,)')

In [6]:
np_sort([[1, 2], [3, 4]], axis=2)

ValueError: axis out of range

In [7]:
np_sort([[1, 2], [3, 4]], axis=0)

NotImplementedError: Only sorting along the last axis is implemented

In [8]:
np_sort([1, "a"])

TypeError: '<=' not supported between instances of 'int' and 'str'

In [9]:
np_sort([3, 1, 2], kind='unknown')

ValueError: Unsupported sort kind: unknown